##L12: Structured Streaming: triggers, checkpoints and progress

###Goal
Understand the stream lifecycle by running a tiny stream between two Delta tables.

###Step 1. Create a small source table

In [0]:
%sql
CREATE OR REPLACE TABLE da.default.stream_src 
AS 
SELECT * FROM da.silver.yellow_trips LIMIT 1000
;

DROP TABLE IF EXISTS da.default.stream_dst;

In [0]:
%sql
SELECT *
FROM da.default.stream_src 
LIMIT 10
;

###Step 2. Stream the source into a destination

In [0]:
base = "/Volumes/da/bronze/landing/_meta/stream_demo"
def run_stream():
    q = (spark.readStream.table("da.default.stream_src")
         .writeStream.option("checkpointLocation", f"{base}/checkpoint")
         .trigger(availableNow=True)
         .toTable("da.default.stream_dst"))
    q.awaitTermination()
    return q
q = run_stream()
print(spark.table("da.default.stream_dst").count())

###Purpose of base
base is just a folder path in a Unity Catalog Volume where all the bookkeeping files for the stream live. The line: checkpointLocation = f"{base}/checkpoint"

tells Spark to persist the stream's state there. Note that it has nothing to do with the data itself — the trip rows go into da.default.stream_dst (a Delta table). The checkpoint directory stores only metadata about the stream's progress.

How the checkpoint works
When the stream runs, Spark writes small files under the checkpoint folder, mainly:

offsets/ – a numbered log of which batches of source data have been read. Batch 0, batch 1, batch 2... Each file says "everything up to here has been ingested."
commits/ – a marker confirming each batch was fully written to the destination. A batch is committed only after the Delta write succeeds.
state/ and metadata/ – state for stateful operations (aggregations, dedup, watermarks) and the stream's configuration identity.

###Step 3. Look at what the stream stored

Find offsets, commits and sources. lastProgress shows numInputRows and the batch id.

In [0]:
for f in dbutils.fs.ls(f"{base}/checkpoint"):
    print(f.name)
print(q.lastProgress)

###Step 4. Append to the source and run again
Only the new rows should flow.

In [0]:
%sql
INSERT INTO da.default.stream_src SELECT * FROM da.silver.yellow_trips LIMIT 100;

In [0]:
q = run_stream()
print(q.lastProgress["numInputRows"])
print(spark.table("da.default.stream_dst").count())

###Step 5. Break it on purpose
Delete the checkpoint folder and run again. What happens to the destination row count, and why?

In [0]:
dbutils.fs.rm(f"{base}/checkpoint", True)
run_stream()
print(spark.table("da.default.stream_dst").count())

Without a checkpoint the stream reprocesses the whole source and appends it again, so rows are duplicated. This is why checkpoints must be kept with the target.

###Step 6. Clean up

In [0]:
%sql
DROP TABLE IF EXISTS da.default.stream_src;
DROP TABLE IF EXISTS da.default.stream_dst;